[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badaouihakimou/machine-learning-notebooks/blob/main/09_programmation_orientee_objet.ipynb)


# La programmation orientée objet

Depuis le début de ces notebooks, on écrit des choses comme `liste.append(4)`,
`data.dropna()`, `tableau.shape`. Ce point entre le nom et ce qui suit n'a jamais
été expliqué.

Ce notebook répond à cette question. En Python, presque tout est un objet :
une liste, un DataFrame, un tableau NumPy, une figure Matplotlib. Comprendre
comment ils sont construits, c'est comprendre comment lire une documentation.

## Le plan

| Section | Le sujet |
|---|---|
| 1 | Objets, attributs et méthodes : ce qu'on utilise déjà |
| 2 | Écrire sa première classe |
| 3 | `self`, la variable qui déroute |
| 4 | Attribut d'instance ou de classe : le piège |
| 5 | `__repr__` et les méthodes spéciales |
| 6 | L'héritage |
| 7 | Lire une documentation |

## Deux pièges annoncés

Un attribut défini au niveau de la classe est partagé par toutes les
instances. Si c'est une liste, modifier l'une modifie toutes les autres.

Et une méthode qui modifie l'objet ne doit rien renvoyer, sous peine de laisser
croire qu'elle produit une copie.

Prérequis : les notebooks 02 à 08.

## 1. Ce qu'on utilise déjà sans le savoir

Prenons un tableau NumPy.

In [4]:
import numpy as np

tableau = np.array([1, 2, 3])

print(type(tableau))

<class 'numpy.ndarray'>


`ndarray` est le nom de sa classe. Le tableau lui-même est une instance
de cette classe.

L'analogie habituelle : la classe est le plan de construction, l'instance est
l'objet construit. Un même plan produit autant d'objets qu'on veut, chacun avec
ses propres valeurs.

### Attributs et méthodes

Le point donne accès à deux choses différentes.

In [5]:
print('Attributs : des données, sans parenthèses')
print('size  :', tableau.size)
print('shape :', tableau.shape)
print('dtype :', tableau.dtype)

print()
print('Méthodes : des actions, avec parenthèses')
print('sum() :', tableau.sum())
print('max() :', tableau.max())
print('mean():', tableau.mean())

Attributs : des données, sans parenthèses
size  : 3
shape : (3,)
dtype : int64

Méthodes : des actions, avec parenthèses
sum() : 6
max() : 3
mean(): 2.0


| | Ce que c'est | Écriture |
|---|---|---|
| Attribut | une donnée que l'objet porte | `objet.attribut` |
| Méthode | une action que l'objet sait faire | `objet.methode()` |

Oublier les parenthèses sur une méthode ne lève pas d'erreur ça renvoie la
fonction elle-même, ce qui est rarement ce qu'on veut :

In [6]:
print(tableau.sum) # la méthode, pas le résultat
print(tableau.sum()) # le résultat

<built-in method sum of numpy.ndarray object at 0x7c24ed16a9d0>
6


C'est une erreur silencieuse classique. Si un affichage montre quelque chose
comme `<built-in method sum of numpy.ndarray>`, tu as oublié les parenthèses.

### Découvrir ce qu'un objet sait faire

`dir()` liste tout ce qui est accessible.

In [7]:
tableau

array([1, 2, 3])

In [8]:
publics = [nom for nom in dir(tableau) if not nom.startswith('_')]
print(len(publics), 'éléments publics')
print(publics[:20])

74 éléments publics
['T', 'all', 'any', 'argmax', 'argmin', 'argpartition', 'argsort', 'astype', 'base', 'byteswap', 'choose', 'clip', 'compress', 'conj', 'conjugate', 'copy', 'ctypes', 'cumprod', 'cumsum', 'data']


Le filtre sur `_` écarte les noms techniques, sur lesquels on reviendra en
section 5.

Et `help()` donne la documentation :

```python
help(tableau.sum)
```

Dans un notebook, `tableau.sum?` fait la même chose en plus court. C'est le
réflexe à prendre plutôt que de chercher sur internet.

In [9]:
help(tableau.sum)

Help on built-in function sum:

sum(...) method of numpy.ndarray instance
    a.sum(axis=None, dtype=None, out=None, keepdims=False, initial=0, where=True)

    Return the sum of the array elements over the given axis.

    Refer to `numpy.sum` for full documentation.

    See Also
    --------
    numpy.sum : equivalent function



In [10]:
tableau.sum?

## 2. Écrire sa première classe

Le mot-clé est `class`. Par convention, le nom s'écrit en PascalCase :
majuscule à chaque mot, sans tiret bas. C'est ce qui distingue visuellement une
classe d'une fonction.

In [11]:
class Vehicule:
    """Un véhicule, caractérisé par sa couleur, sa vitesse et son nombre de roues."""

    def __init__(self, couleur='noire', vitesse=0, roues=4):
        """Construit un véhicule."""
        self.couleur = couleur
        self.vitesse = vitesse
        self.roues = roues

    def accelerer(self, gain):
        """Augmente la vitesse. Modifie l'objet."""
        self.vitesse += gain

    def stop(self):
        """Ramène la vitesse à zéro."""
        self.vitesse = 0

    def afficher(self):
        """Affiche l'état du véhicule."""
        print(f'couleur : {self.couleur}')
        print(f'roues : {self.roues}')
        print(f'vitesse : {self.vitesse}')

Créer une instance revient à appeler la classe comme une fonction.

In [12]:
voiture_1 = Vehicule(couleur='rouge')

voiture_1.afficher()

couleur : rouge
roues : 4
vitesse : 0


### Chaque instance a ses propres valeurs

C'est le point essentiel : deux objets construits depuis la même classe sont
indépendants.

In [13]:
voiture_1 = Vehicule(couleur='rouge')
voiture_2 = Vehicule(couleur='bleue', roues=2)

voiture_1.accelerer(100)

print('voiture 1 :', voiture_1.couleur, voiture_1.vitesse, 'km/h')
print('voiture 2 :', voiture_2.couleur, voiture_2.vitesse, 'km/h')
print('même objet ?', voiture_1 is voiture_2)

voiture 1 : rouge 100 km/h
voiture 2 : bleue 0 km/h
même objet ? False


Accélérer la première ne touche pas la seconde. Chacune porte ses propres
attributs.

### Modifier directement un attribut

Rien n'empêche d'écrire dans un attribut de l'extérieur.

In [14]:
voiture_1.vitesse = 999
print(voiture_1.vitesse)

voiture_1.couleur_preferee = 'vert' # on peut même en créer un nouveau
print(voiture_1.couleur_preferee)
print(hasattr(voiture_2, 'couleur_preferee'))

999
vert
False


Python n'a pas de véritable protection des attributs. La convention est le tiret
bas initial : `self._interne` signale « ne touche pas », sans l'interdire
techniquement.

C'est un choix de philosophie du langage, souvent résumé par « nous sommes tous
des adultes consentants ». D'autres langages verrouillent réellement.

### Une méthode qui modifie ne renvoie rien

Notre `accelerer` modifie `self.vitesse` et ne fait pas de `return`.

In [15]:
resultat = voiture_1.accelerer(10)
print('retour :', resultat)

retour : None


`None`, comme `liste.append()` ou `liste.sort()`. C'est la convention Python vue
au notebook 04 : une méthode qui modifie sur place renvoie `None`, une
fonction qui produit un nouvel objet le renvoie.

Cette règle explique enfin la différence :

```python
liste.sort() # méthode, modifie, renvoie None
sorted(liste) # fonction, ne modifie pas, renvoie une copie

data.dropna() # renvoie une COPIE, d'où le data = data.dropna()
```

Pandas fait exception à la convention : ses méthodes renvoient des copies. C'est
pour ça qu'il faut réassigner.

## 3. self, la variable qui déroute

`self` désigne l'instance sur laquelle la méthode est appelée. Python le
passe automatiquement, sans qu'on l'écrive à l'appel.

In [16]:
class Demo:
    def __init__(self, nom):
        self.nom = nom

    def qui_suis_je(self):
        print('self est :', self)
        print('mon nom  :', self.nom)


a = Demo('objet A')
b = Demo('objet B')

a.qui_suis_je()
print()
b.qui_suis_je()

self est : <__main__.Demo object at 0x7c24ed228680>
mon nom  : objet A

self est : <__main__.Demo object at 0x7c24ed2286b0>
mon nom  : objet B


Deux adresses mémoire différentes : `self` n'est pas la même chose selon l'objet
appelant.

### La preuve que Python le passe tout seul

Ces deux écritures sont strictement équivalentes :

In [17]:
a.qui_suis_je() # la forme normale
Demo.qui_suis_je(a) # ce que Python fait réellement

self est : <__main__.Demo object at 0x7c24ed228680>
mon nom  : objet A
self est : <__main__.Demo object at 0x7c24ed228680>
mon nom  : objet A


`a.qui_suis_je()` est traduit en `Demo.qui_suis_je(a)`. L'objet devient le
premier argument, celui qu'on appelle `self`.

C'est pour cela que toute méthode doit déclarer `self` en premier paramètre.
L'oublier produit une erreur déroutante :

In [18]:
class Casse:
    def methode_sans_self():
        print('bonjour')

try:
    Casse().methode_sans_self()
except TypeError as e:
    print('TypeError :', e)

TypeError : Casse.methode_sans_self() takes 0 positional arguments but 1 was given


Le message dit qu'on a passé un argument alors que la méthode n'en attend aucun
cet argument invisible, c'est l'instance.

Le nom `self` n'est pas imposé par le langage. On pourrait écrire `moi` ou
`obj`. Mais c'est une convention universelle, et s'en écarter rend le code
illisible pour tout le monde.

### Sans self, la variable est locale

Une erreur fréquente : oublier le `self.` dans `__init__`.

In [19]:
class Mauvais:
    def __init__(self, nom):
        nom = nom  # variable locale, perdue à la fin de __init__

class Bon:
    def __init__(self, nom):
        self.nom = nom  # attribut de l'instance, conservé


try:
    print(Mauvais('test').nom)
except AttributeError as e:
    print('AttributeError :', e)

print(Bon('test').nom)

AttributeError : 'Mauvais' object has no attribute 'nom'
test


Sans `self.`, on crée une simple variable locale qui disparaît à la fin de la
fonction. C'est la portée des variables du notebook 02.

## 4. Attribut d'instance ou de classe

Un attribut peut être défini à deux endroits, et la différence est piégeuse.

In [20]:
class Voiture:
    nb_roues = 4 # attribut de CLASSE, écrit hors de __init__

    def __init__(self, couleur):
        self.couleur = couleur # attribut d'INSTANCE


a = Voiture('rouge')
b = Voiture('bleue')

print('couleurs :', a.couleur, b.couleur) # différentes
print('roues :', a.nb_roues, b.nb_roues) # la même valeur, partagée
print('via la classe :', Voiture.nb_roues)

couleurs : rouge bleue
roues : 4 4
via la classe : 4


L'attribut de classe est partagé par toutes les instances. Il est utile pour
une constante commune, ou pour un compteur.

Avec une valeur immuable, tout va bien : réaffecter sur une instance crée un
attribut propre à celle-ci.

In [21]:
a.nb_roues = 6 # crée un attribut d'instance sur a

print('a :', a.nb_roues)
print('b :', b.nb_roues) # inchangé
print('classe :', Voiture.nb_roues)

a : 6
b : 4
classe : 4


### Le piège : un attribut de classe mutable

Avec une liste, tout change.

In [22]:
class Garage:
    voitures = [] # attribut de classe, mutable

    def __init__(self, nom):
        self.nom = nom

    def ajouter(self, voiture):
        self.voitures.append(voiture)

g1 = Garage('Paris')
g2 = Garage('Lyon')

g1.ajouter('Clio')

print('garage 1 :', g1.voitures)
print('garage 2 :', g2.voitures)
print('même liste ?', g1.voitures is g2.voitures)

garage 1 : ['Clio']
garage 2 : ['Clio']
même liste ? True


La Clio apparaît dans les deux garages.

`self.voitures.append(...)` ne réaffecte rien : il modifie la liste partagée
par toutes les instances. Contrairement à `a.nb_roues = 6`, qui créait un nouvel
attribut.

C'est le même mécanisme que le défaut mutable d'une fonction au notebook 02, et
que `dict.fromkeys(cles, [])` au notebook 05. Une valeur mutable créée une seule
fois puis partagée.

La correction : créer la liste dans `__init__`.

In [23]:
class Garage:
    def __init__(self, nom):
        self.nom = nom
        self.voitures = [] # une liste neuve pour chaque instance

    def ajouter(self, voiture):
        self.voitures.append(voiture)


g1 = Garage('Paris')
g2 = Garage('Lyon')

g1.ajouter('Clio')

print('garage 1 :', g1.voitures)
print('garage 2 :', g2.voitures)
print('même liste ?', g1.voitures is g2.voitures)

garage 1 : ['Clio']
garage 2 : []
même liste ? False


La règle : tout ce qui est propre à une instance se définit dans `__init__`,
avec `self.`. Les attributs de classe sont réservés aux constantes partagées.

Un usage légitime de l'attribut de classe, en revanche : compter les instances.

In [24]:
class Compteur:
    nombre = 0  # entier, immuable : pas de piège

    def __init__(self):
        Compteur.nombre += 1 # on cible la classe explicitement

for _ in range(5):
    Compteur()

print('Instances créées :', Compteur.nombre)

Instances créées : 5


Note le `Compteur.nombre += 1` et non `self.nombre += 1`. Le second créerait un
attribut d'instance et le compteur resterait bloqué à 1 un piège de plus.

## 5. Les méthodes spéciales

Les noms entourés de doubles tirets bas `__init__`, `__len__`, `__repr__`
sont appelés automatiquement par Python dans certaines situations. On les
surnomme les *dunder methods*.

`__init__` en est une : elle s'exécute à la création de l'objet.

### __repr__ : afficher son objet

Sans elle, l'affichage d'un objet est inutilisable.

In [25]:
class Sans:
    def __init__(self, couleur):
        self.couleur = couleur

print(Sans('rouge'))

In [26]:
class Avec:
    def __init__(self, couleur):
        self.couleur = couleur

    def __repr__(self):
        return f'Avec(couleur={self.couleur!r})'

objet = Avec('rouge')
print(objet)
objet  # même dans une cellule sans print

Avec(couleur='rouge')


Avec(couleur='rouge')

Le `!r` dans la f-string affiche la valeur avec ses guillemets, ce qui distingue
une chaîne d'un nombre.

La convention est que `__repr__` renvoie une chaîne qui, idéalement, permettrait
de recréer l'objet. C'est ce que fait NumPy :

```python
np.array([1, 2, 3]) # affiche array([1, 2, 3])
```

`__str__` existe aussi, pour un affichage destiné à l'utilisateur final. Si l'on
n'en définit qu'une, `__repr__` est le bon choix : elle sert de repli.

### Les autres dunder utiles

In [27]:
class Panier:
    def __init__(self, articles=None):
        self.articles = articles if articles is not None else []

    def __repr__(self):
        return f'Panier({self.articles})'

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, i):
        return self.articles[i]

    def __contains__(self, article):
        return article in self.articles

    def __add__(self, autre):
        return Panier(self.articles + autre.articles)


p1 = Panier(['pain', 'lait'])
p2 = Panier(['oeufs'])

print(p1)
print('len       :', len(p1))
print('p1[0]     :', p1[0])
print('in        :', 'lait' in p1)
print('addition  :', p1 + p2)

for article in p1:  # __getitem__ suffit pour l'itération
    print(' -', article)

Panier(['pain', 'lait'])
len       : 2
p1[0]     : pain
in        : True
addition  : Panier(['pain', 'lait', 'oeufs'])
 - pain
 - lait


En définissant ces quelques méthodes, l'objet se comporte comme une structure
native de Python : on peut le mesurer, l'indexer, le parcourir, l'additionner.

C'est exactement ce qui permet d'écrire `len(df)`, `df['colonne']` ou
`tableau1 + tableau2`. Ces bibliothèques définissent les mêmes méthodes.

Note : le `articles if articles is not None else []` reprend la parade du
défaut mutable du notebook 02. Écrire `def __init__(self, articles=[])` serait le
même bug.

## 6. L'héritage

Une classe peut reprendre les attributs et méthodes d'une autre, en n'écrivant
que ce qui diffère.

In [28]:
class VoitureElectrique(Vehicule):
    """Un véhicule électrique, avec une autonomie qui diminue à l'usage."""

    def __init__(self, couleur='noire', vitesse=0, roues=4, autonomie=100):
        super().__init__(couleur, vitesse, roues) # appelle le parent
        self.autonomie = autonomie

    def accelerer(self, gain):
        """Accélère, et consomme de l'autonomie."""
        super().accelerer(gain)
        self.autonomie -= 0.1 * self.vitesse

    def afficher(self):
        super().afficher()
        print(f'autonomie : {self.autonomie:.1f}')


voiture = VoitureElectrique(couleur='blanche')
voiture.afficher()

print()
voiture.accelerer(50)
voiture.afficher()

couleur : blanche
roues : 4
vitesse : 0
autonomie : 100.0

couleur : blanche
roues : 4
vitesse : 50
autonomie : 95.0


`super()` désigne la classe parente. Il permet de réutiliser son code plutôt que
de le recopier.

Dans `accelerer`, on appelle d'abord la version du parent qui met à jour la
vitesse puis on ajoute le comportement spécifique.

La méthode `stop` n'a pas été réécrite, et pourtant elle fonctionne : elle
est héritée telle quelle.

In [29]:
voiture.stop()
print('après stop :', voiture.vitesse)

après stop : 0


### Vérifier le type

`isinstance` tient compte de l'héritage, contrairement à `type`.

In [30]:
print('isinstance VoitureElectrique :', isinstance(voiture, VoitureElectrique))
print('isinstance Vehicule :', isinstance(voiture, Vehicule))
print('type == Vehicule :', type(voiture) == Vehicule)
print()
print('issubclass :', issubclass(VoitureElectrique, Vehicule))

isinstance VoitureElectrique : True
isinstance Vehicule : True
type == Vehicule : False

issubclass : True


Un objet `VoitureElectrique` est un `Vehicule`. C'est le sens de l'héritage,
et c'est pour ça qu'`isinstance` est préférable à `type(x) == Y`.

Une curiosité qui en découle :

In [31]:
print('True est un int :', isinstance(True, int))
print('True + True =', True + True)

True est un int : True
True + True = 2


`bool` hérite de `int` en Python, ce qui explique que `sum` compte les `True`
comme des 1 l'idiome du notebook 07.

### Quand ne pas utiliser l'héritage

L'héritage exprime une relation « est un ». Une voiture électrique est un
véhicule : la relation tient.

Quand la relation est « a un » une voiture a un moteur il vaut mieux la
composition, c'est-à-dire mettre un objet moteur en attribut :

```python
class Voiture:
    def __init__(self, moteur):
        self.moteur = moteur
```

L'héritage sur plusieurs niveaux devient vite difficile à suivre. Deux niveaux
suffisent presque toujours.

## 7. Lire une documentation

C'est l'intérêt pratique de tout ce notebook. La documentation de NumPy, Pandas
ou scikit-learn décrit des classes, et la structure est toujours la même.

Prenons `ndarray`, dont la documentation est sur
[numpy.org](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html).

In [32]:
tableau = np.array([[1, 2, 3], [4, 5, 6]])

print('Attributs')
for nom in ['shape', 'ndim', 'size', 'dtype', 'T']:
    print(f'{nom:<7} : {getattr(tableau, nom)}')

Attributs
shape   : (2, 3)
ndim    : 2
size    : 6
dtype   : int64
T       : [[1 4]
 [2 5]
 [3 6]]


In [33]:
print(' Méthodes ')
print(' sum() :', tableau.sum())
print(' sum(axis=0) :', tableau.sum(axis=0))
print(' reshape :', tableau.reshape(3, 2).tolist())
print(' copy is new :', tableau.copy() is tableau)

 Méthodes 
 sum() : 21
 sum(axis=0) : [5 7 9]
 reshape : [[1, 2], [3, 4], [5, 6]]
 copy is new : False


Une documentation de classe liste toujours ces deux catégories : les attributs et
les méthodes. Savoir les distinguer permet de comprendre à quoi s'attendre une donnée ou une action.

### Le cas de scikit-learn

C'est là que la POO devient concrète pour le machine learning. Tous les modèles
suivent la même interface :

```python
modele = LinearRegression() # création de l'instance
modele.fit(X, y) # méthode : apprend, modifie l'objet
modele.predict(X_test) # méthode : produit un résultat
modele.coef_ # attribut : les coefficients appris
```

Trois choses à remarquer.

Les hyperparamètres passent au constructeur `LinearRegression(fit_intercept=False)`.
Ce sont les choix qu'on fait avant l'entraînement.

Les attributs se terminant par un tiret bas `coef_`, `intercept_` sont
ceux calculés pendant `fit`. Y accéder avant lève une erreur. C'est une
convention de scikit-learn, pas de Python.

Et `fit` modifie l'objet puis renvoie `self`, ce qui permet d'enchaîner :
`modele.fit(X, y).predict(X)`.

Cette régularité vient de l'héritage : tous les modèles descendent d'une classe
de base commune. C'est pour ça qu'on peut remplacer une régression logistique par
une forêt aléatoire en changeant une seule ligne.

## 8. Mémo

### Le vocabulaire

| Terme | Définition |
|---|---|
| Classe | le plan de construction |
| Instance | un objet construit depuis ce plan |
| Attribut | une donnée portée par l'objet, sans parenthèses |
| Méthode | une action, avec parenthèses |
| `self` | l'instance sur laquelle la méthode est appelée |
| Héritage | reprendre le contenu d'une autre classe |

### La structure

```python
class MaClasse(ClasseParente):
    """Docstring."""

    attribut_de_classe = 0 # partagé par toutes les instances

    def __init__(self, valeur):
        super().__init__() # si héritage
        self.attribut = valeur # propre à l'instance

    def methode(self, argument):
        self.attribut += argument # modifie, ne renvoie rien

    def __repr__(self):
        return f'MaClasse({self.attribut})'
```

### Les pièges

| Situation | Ce qui se passe |
|---|---|
| Attribut de classe mutable | partagé par toutes les instances |
| `nom = nom` sans `self.` | variable locale perdue |
| Méthode sans `self` en paramètre | `TypeError` à l'appel |
| `self.compteur += 1` sur un attribut de classe | crée un attribut d'instance |
| Méthode appelée sans parenthèses | renvoie la fonction, pas le résultat |
| Pas de `__repr__` | affichage inutilisable |
| `def __init__(self, l=[])` | défaut mutable partagé |

## 9. Exercices

**Exercice 1**

Écris une classe `CompteBancaire` avec un solde, un titulaire, et un historique
des opérations. Elle doit avoir des méthodes `deposer`, `retirer` et
`historique`, un `__repr__` utile, et refuser un retrait supérieur au solde.

Crée deux comptes et vérifie que leurs historiques sont bien indépendants.

**Exercice 2**

Écris une classe `Statistiques` qui reçoit une liste de nombres et expose la
moyenne, la médiane et l'écart-type. Ajoute `__len__` pour connaître le nombre de
valeurs, et `__repr__`.

Puis crée une sous-classe `StatistiquesPonderees` qui accepte des poids, en
réutilisant ce qui peut l'être avec `super()`.

**Exercice 3**

Cette classe contient deux pièges de ce notebook :

```python
class Panier:
    articles = []

    def __init__(self, proprietaire):
        proprietaire = proprietaire

    def ajouter(self, article):
        self.articles.append(article)
        return self.articles
```

Crée deux paniers, ajoute un article à chacun, et montre les deux problèmes.
Explique-les, puis corrige la classe.

## Pour continuer

Le notebook suivant ouvre NumPy et tout ce qui a été vu ici sert directement.
`np.array` est une classe, `.shape` un attribut, `.reshape()` une méthode.

En Pandas, `DataFrame` et `Series` sont deux classes, et l'essentiel du travail
consiste à appeler leurs méthodes. Savoir lire leur documentation change
complètement la façon d'apprendre : au lieu de chercher un exemple sur internet,
on regarde ce que l'objet sait faire.

In [38]:
# Exercice 1

In [35]:
class CompteBancaire:
    """Un compte bancaire, avec son historique d'opérations."""

    taux_interet = 0.02 # attribut de CLASSE : immuable, partagé, sans danger

    def __init__(self, titulaire, solde=0):
        self.titulaire = titulaire
        self.solde = solde
        self.operations = [] # dans __init__ : propre à l'instance
        if solde:
            self.operations.append(('ouverture', solde))

    def __repr__(self):
        return f'CompteBancaire({self.titulaire!r}, solde={self.solde:.2f})'

    def deposer(self, montant):
        """Ajoute un montant au solde. Modifie le compte."""
        if montant <= 0:
            raise ValueError('Le montant doit être positif')
        self.solde += montant
        self.operations.append(('dépôt', montant))

    def retirer(self, montant):
        """Retire un montant. Refuse si le solde est insuffisant."""
        if montant <= 0:
            raise ValueError('Le montant doit être positif')
        if montant > self.solde:
            raise ValueError(f'Solde insuffisant : {self.solde:.2f} € disponibles')
        self.solde -= montant
        self.operations.append(('retrait', -montant))

    def historique(self):
        """Affiche toutes les opérations et le solde final."""
        print(f'--- {self.titulaire} ---')
        for operation, montant in self.operations:
            print(f'{operation:<12} {montant:>+10.2f}')
        print(f'{"solde":<12} {self.solde:>10.2f}')

In [36]:
compte_a = CompteBancaire('Ali', 100)
compte_b = CompteBancaire('Sara')

compte_a.deposer(50)
compte_b.deposer(20)

print(compte_a)
print(compte_b)
print('Historiques indépendants ?', compte_a.operations is not compte_b.operations)

compte_a.historique()

CompteBancaire('Ali', solde=150.00)
CompteBancaire('Sara', solde=20.00)
Historiques indépendants ? True
--- Ali ---
ouverture       +100.00
dépôt            +50.00
solde            150.00


In [37]:
try:
    compte_b.retirer(1000)
except ValueError as e:
    print('ValueError :', e)

print('Solde inchangé :', compte_b.solde)

ValueError : Solde insuffisant : 20.00 € disponibles
Solde inchangé : 20


In [40]:
total = sum(m for _, m in compte_a.operations)
print('Cohérence :', abs(total - compte_a.solde) < 1e-9)

Cohérence : True


In [41]:
# Exercice 2

In [42]:
import statistics

class Statistiques:
    """Calcule des statistiques descriptives sur une liste de nombres."""

    def __init__(self, valeurs):
        if not valeurs:
            raise ValueError('La liste ne peut pas être vide')
        self.valeurs = list(valeurs) # list() : une COPIE

    def __len__(self):
        return len(self.valeurs)

    def __repr__(self):
        return f'Statistiques(n={len(self)}, moyenne={self.moyenne():.2f})'

    def moyenne(self):
        return sum(self.valeurs) / len(self.valeurs)

    def mediane(self):
        return statistics.median(self.valeurs)

    def ecart_type(self):
        if len(self) < 2:
            return 0.0   # stdev exige au moins 2 valeurs
        return statistics.stdev(self.valeurs)

In [43]:
s = Statistiques([10, 12, 14, 20])

print(s)
print('len :', len(s))
print('moyenne :', s.moyenne())
print('médiane :', s.mediane())
print('écart-type :', round(s.ecart_type(), 3))

Statistiques(n=4, moyenne=14.00)
len : 4
moyenne : 14.0
médiane : 13.0
écart-type : 4.32


In [44]:
donnees = [1, 2, 3]
s = Statistiques(donnees)
donnees.append(1000)
print(s.moyenne()) # affecté si on n'a pas copié

2.0


In [45]:
class StatistiquesPonderees(Statistiques):
    """Statistiques avec un poids par valeur."""

    def __init__(self, valeurs, poids):
        super().__init__(valeurs) # réutilise le contrôle et la copie
        if len(poids) != len(valeurs):
            raise ValueError('valeurs et poids doivent avoir la même longueur')
        self.poids = list(poids)

    def moyenne(self):
        """Moyenne pondérée : redéfinit celle du parent."""
        return sum(v * p for v, p in zip(self.valeurs, self.poids)) / sum(self.poids)

    def __repr__(self):
        return f'StatistiquesPonderees(n={len(self)}, moyenne={self.moyenne():.2f})'

In [46]:
sp = StatistiquesPonderees([10, 20], [1, 3])

print(sp)
print('moyenne pondérée :', sp.moyenne()) # 17.5, tirée vers 20
print('médiane héritée :', sp.mediane()) # 15.0
print('len hérité :', len(sp))

StatistiquesPonderees(n=2, moyenne=17.50)
moyenne pondérée : 17.5
médiane héritée : 15.0
len hérité : 2


In [47]:
# Exercice 3

In [48]:
class Panier:
    articles = []

    def __init__(self, proprietaire):
        proprietaire = proprietaire

    def ajouter(self, article):
        self.articles.append(article)
        return self.articles

In [49]:
p1 = Panier('Ali')
p2 = Panier('Sara')

p1.ajouter('pain')
p2.ajouter('lait')

print('panier 1 :', p1.articles)
print('panier 2 :', p2.articles)
print('même liste ?', p1.articles is p2.articles)

panier 1 : ['pain', 'lait']
panier 2 : ['pain', 'lait']
même liste ? True


In [50]:
print('au niveau de la classe :', Panier.articles)

au niveau de la classe : ['pain', 'lait']


In [51]:
try:
    print(p1.proprietaire)
except AttributeError as e:
    print('AttributeError :', e)

AttributeError : 'Panier' object has no attribute 'proprietaire'


In [52]:
resultat = p1.ajouter('beurre')
print('même objet ?', resultat is p1.articles) # True

même objet ? True


In [53]:
class Panier:
    """Un panier d'articles, propre à un propriétaire."""

    def __init__(self, proprietaire):
        self.proprietaire = proprietaire # self. : attaché à l'instance
        self.articles = [] # dans __init__ : liste neuve

    def __repr__(self):
        return f'Panier({self.proprietaire!r}, {len(self.articles)} articles)'

    def __len__(self):
        return len(self.articles)

    def ajouter(self, article):
        """Ajoute un article. Modifie le panier, ne renvoie rien."""
        self.articles.append(article)

In [54]:
p1 = Panier('Ali')
p2 = Panier('Sara')

p1.ajouter('pain')
p2.ajouter('lait')

print(p1, '->', p1.articles)
print(p2, '->', p2.articles)
print('même liste ?', p1.articles is p2.articles)
print('propriétaire :', p1.proprietaire)

Panier('Ali', 1 articles) -> ['pain']
Panier('Sara', 1 articles) -> ['lait']
même liste ? False
propriétaire : Ali
